<a href="https://colab.research.google.com/github/BastianRu/Deep-Learning-Foundations-From-Scratch-Journal/blob/main/03-Energy-Based_Models_(EBM)_Variant/EBM_Variant_Bengio03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#load all the data-set
!wget https://raw.githubusercontent.com/karpathy/makemore/master/names.txt -O names.txt #source

#Libraries
import torch, random
import torch.nn.functional as F

#Stoi and Itos dictionaries
chars = ['.', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']

stoi  = { s:i for i, s in enumerate(chars)}
itos = { i:s for s, i in stoi.items()}

--2026-01-27 22:33:40--  https://raw.githubusercontent.com/karpathy/makemore/master/names.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.111.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 228145 (223K) [text/plain]
Saving to: ‘names.txt’

names.txt           100%[===================>] 222.80K  --.-KB/s    in 0.03s   

2026-01-27 22:33:40 (7.81 MB/s) - ‘names.txt’ saved [228145/228145]



In [ ]:
#Build the data set
with open('names.txt', 'r') as f:
    names = f.read().splitlines()

block_size = 3
xs, ys = [], []

for w in names:
  context = [0] * block_size
  for ch in w + '.':
    xs.append(context)
    ys.append(stoi[ch])
   # print(f'{''.join([itos[ix] for ix in context])} -> {ch}' )
    context = context[1:] + [stoi[ch]]


xs = torch.tensor(xs) #switch
ys = torch.tensor(ys)

random.seed(42)
random.shuffle(names) #Shuffle all the data-set (this allows us to get a non-biased data)

#Let's create the training, dev and test split
Xtrain, Ytrain = [], [] #80%
Xdev, Ydev = [], [] #10%
Xtest, Ytest = [], [] #10%

Xtrain, Ytrain = torch.tensor(xs[:int(len(xs)*0.8)]), torch.tensor(ys[:int(len(xs)*0.8)])
Xdev, Ydev = torch.tensor(xs[int(len(xs)*0.8): int(len(xs)*0.9)]), torch.tensor(ys[int(len(xs)*0.8): int(len(xs)*0.9)])
Xtest, Ytest = torch.tensor(xs[int(len(xs)*0.9): len(xs)]), torch.tensor(ys[int(len(xs)*0.9): len(xs)])

Xtrain.shape, Xdev.shape, Xtest.shape

#xs, ys

/tmp/ipython-input-858786310.py:28: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  Xtrain, Ytrain = torch.tensor(xs[:int(len(xs)*0.8)]), torch.tensor(ys[:int(len(xs)*0.8)])
/tmp/ipython-input-858786310.py:29: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  Xdev, Ydev = torch.tensor(xs[int(len(xs)*0.8): int(len(xs)*0.9)]), torch.tensor(ys[int(len(xs)*0.8): int(len(xs)*0.9)])
/tmp/ipython-input-858786310.py:30: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  Xtest, Ytest = torch.tensor(xs[int(len(xs)*0.9): len(xs)]), torch.tensor(ys[int(len(xs)*

(torch.Size([182516, 3]), torch.Size([22815, 3]), torch.Size([22815, 3]))

In [ ]:
#Initialize the network (NOW WITH ENERGY BASED MODEL)
g=torch.Generator().manual_seed(73)

emb_dim= 8 #embedding number of dimensions
C = torch.randn((27, emb_dim), generator=g)

nin = (block_size + 1) * emb_dim #+1 because now the input brings the label
nout = 300 #Number of neurons for the hidden layer

W1 = torch.randn((nin, nout), generator=g)
b1 = torch.randn(nout, generator=g)

v = torch.randn((nout), generator=g)
b2 = torch.randn(27, generator=g)

parameters = [C, W1, v, b1, b2]

for p in parameters:
  p.requires_grad = True

batch_size = 32 #batch size for training



In [ ]:
#Forward
#inputs = [torch.cat((xs[0], torch.tensor([ix]))) for ix in range(27)]

inputs = torch.tensor([ xs[i].tolist() + [ix] for i in range(len(xs)) for ix in range(27) ]) #best way i managed to found

emb_biases = b2[inputs]

#Now view is -1, 8 (2 numbers for each character, 4 characters per example)
hidden = (C[inputs].view(-1, nin) @ W1 + b1).tanh()

#hidden @ v = (N, 100) @ (1, 100)
pre_energy = hidden @ v
bias_sum = emb_biases.sum(dim=1) #keepdim neccesary? No. We're summing vectors elementwise

energy = (pre_energy + bias_sum).view(-1, 27) #elementwise sum and reshaping from (num_examples * 27) to (num_examples, 27)

#it's the negative of energy because we expect the energy to be low when the sequence is likely
logits = -energy
loss = F.cross_entropy(logits, ys)

loss

In [ ]:
#Training cycle
def train_model(batch_size, steps, lr):
  for _ in range(steps):

    #batching
    idxs = torch.randint(0, Xtrain.shape[0], (batch_size,))

    #forward pass
    inputs = torch.tensor([ Xtrain[idx].tolist() + [ix] for i, idx in zip(range(batch_size), idxs) for ix in range(27) ])
    emb_biases = b2[inputs]
    #print(inputs[:27])

    hidden = (C[inputs].view(-1, nin) @ W1 + b1).tanh()

    #print(C[inputs].view(-1, 8))

    pre_energy = hidden @ v
    bias_sum = emb_biases.sum(dim=1)

    energy = (pre_energy + bias_sum).view(-1, 27)
    logits = -energy

    #print(logits.shape)

    #loss
    loss = F.cross_entropy(logits, Ytrain[idxs])
    #print(loss.item())

    #backward pass
    for p in parameters: #zero-gradient
      p.grad = None
    loss.backward() #backprop

    for p in parameters:
      p.data += -lr*p.grad

  return loss.item()

train_model(32, steps=5000, lr=0.01)


In [ ]:
#Dev loss
inputs = torch.tensor([ Xdev[i].tolist() + [ix] for i in range(len(Xdev)) for ix in range(27)])
emb_biases = b2[inputs]
hidden = (C[inputs].view(-1, nin) @ W1 + b1).tanh()
pre_energy = hidden @ v
bias_sum = emb_biases.sum(dim=1)
energy = (pre_energy + bias_sum).view(-1, 27)
logits = -energy
loss = F.cross_entropy(logits, Ydev)
loss


In [ ]:
#Sampling for the model

w = []
for i in range(1):
  context = [0] * block_size
  while True:
    ix = 0
    inputs = torch.tensor([ context + [ix] for ix in range(27)])
    print(inputs)
    break

#### WORK IN PROGRESS...